In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# Imports

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, GridSearchCV
from sklearn.decomposition import TruncatedSVD
import os
import wandb
from scipy.stats import uniform, loguniform, randint
import sys
sys.path.append(os.path.join(os.getcwd(), '..'))
from util.preprocessing import TweetPreprocessor 
import joblib
import psutil


# Consts

In [ ]:
MODEL_DIR = os.path.join(os.getcwd(), 'models')
DATASETS_DIR = os.path.join(os.getcwd(), 'datasets')

# Dataset

In [ ]:
train_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_train.csv"))
val_df = pd.read_csv(os.path.join(DATASETS_DIR, "one_tweet_dataset_val.csv"))
train_df = pd.concat([train_df, val_df], ignore_index=True)

In [ ]:
x_train, y_train = train_df["text"], train_df["gender_label"]

# Initiate pipeline

In [ ]:
pipeline = Pipeline([
    ("preprocessor", TweetPreprocessor()),
    ("features", FeatureUnion([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])

# Init wandb

In [ ]:
wandbToken = os.getenv("WANDB_TOKEN")
if not wandbToken:
    raise ValueError("Please set the WANDB_TOKEN environment variable to log results to Weights & Biases.")
wandb.login(key=wandbToken)

# Randomized search

In [ ]:
cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=int(os.getenv("RANDOM_SEED", 880055535)),
)

## TF-idf 

In [ ]:
texts = TweetPreprocessor().fit_transform(x_train)

### Word

In [ ]:
ngram_ranges_word = [(1, 2), (1, 3)]

rnd_params_tfidf_word = {
    "tfidf_word__use_idf": [True, False],
    "tfidf_word__sublinear_tf": [True, False],
    "tfidf_word__norm": ["l1", "l2"],
    "tfidf_word__max_df": uniform(0.6, 0.4),
    "tfidf_word__min_df": uniform(0.001, 0.4),
    "tfidf_word__max_features": randint(5000, 120000),
    "tfidf_word__ngram_range": ngram_ranges_word,
}
rnd_params_tfidf_word

In [ ]:
rnd_search_tfidf_word = RandomizedSearchCV(
    Pipeline([
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
        ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ]),
    rnd_params_tfidf_word, 
    n_iter=500,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_search_tfidf_word.fit(texts, y_train)

In [ ]:
rnd_params_after_tfidf_word = rnd_search_tfidf_word.best_params_
rnd_params_after_tfidf_word

### Char

In [ ]:
ngram_ranges_char = [(2, 4), (3, 5), (4, 6)]

rnd_params_tfidf_char = {
    "tfidf_char__use_idf": [True, False],
    "tfidf_char__sublinear_tf": [True, False],
    "tfidf_char__norm": ["l1", "l2"],
    "tfidf_char__max_df": uniform(0.6, 0.4),
    "tfidf_char__min_df": uniform(0.001, 0.4),
    "tfidf_char__max_features": randint(5000, 120000),
    "tfidf_char__ngram_range": ngram_ranges_char,
}
rnd_params_tfidf_char

In [ ]:
rnd_search_tfidf_char = RandomizedSearchCV(
    Pipeline([
        ("tfidf_char", TfidfVectorizer(analyzer="char")), # type: ignore
        ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ]),
    rnd_params_tfidf_char,
    n_iter=500,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_search_tfidf_char.fit(texts, y_train)

In [ ]:
rnd_params_after_tfidf_char = rnd_search_tfidf_char.best_params_
rnd_params_after_tfidf_char

## SVD

In [ ]:
rnd_svd_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])
rnd_merged_params_for_svd = {
    **{f"features__{k}": v for k, v in rnd_params_after_tfidf_word.items()},
    **{f"features__{k}": v for k, v in rnd_params_after_tfidf_char.items()},
}
print(rnd_merged_params_for_svd)
rnd_svd_pipeline.set_params(**rnd_merged_params_for_svd)

In [ ]:
rnd_svd_params = {
    "svd__n_components": randint(100, 500),
}
rnd_svd_params

In [ ]:
rnd_svd_search = RandomizedSearchCV(
    rnd_svd_pipeline,
    rnd_svd_params, 
    n_iter=500,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_svd_search.fit(texts, y_train)

In [ ]:
rnd_params_after_svd = rnd_svd_search.best_params_
rnd_params_after_svd

## CLF

In [ ]:
rnd_clf_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])
rnd_merged_params_for_clf = {
    **{f"features__{k}": v for k, v in rnd_params_after_tfidf_word.items()},
    **{f"features__{k}": v for k, v in rnd_params_after_tfidf_char.items()},
    **rnd_params_after_svd,
}
print(rnd_merged_params_for_clf)
rnd_clf_pipeline.set_params(**rnd_merged_params_for_clf)

In [ ]:
rnd_clf_params = {
    "clf__C": loguniform(1e-3, 1e3),
}
rnd_clf_params

In [ ]:
rnd_clf_search = RandomizedSearchCV(
    rnd_clf_pipeline,
    rnd_clf_params,
    n_iter=500,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=2,
    random_state=int(os.getenv("RANDOM_SEED", 880055535))
)

In [ ]:
rnd_clf_search.fit(texts, y_train)

## Logging results

In [ ]:
rnd_run = wandb.init(project="who-wrote-it-nlp", name="hyperparam_search", entity="who-wrote-it-nlp", job_type="hyperparam_search_rnd",group="random_search")

rnd_results_df = pd.DataFrame(rnd_clf_search.cv_results_)

for i, row in rnd_results_df.iterrows():
    params = row["params"]
    wandb.log(
        {
            **params,
            "score": row["mean_test_score"],
            "std": row["std_test_score"],
        },
    )

# Log full CV results table for interactive analysis in W&B UI
wandb.log({"rnd_cv_results": wandb.Table(dataframe=rnd_results_df)})

# Auto-log scatter plots for numeric params vs score/std
for p in rnd_params_tfidf_word.keys():
    col = f"param_{p}"
    if col in rnd_results_df.columns and pd.api.types.is_numeric_dtype(rnd_results_df[col]):
        score_plot_df = rnd_results_df[[col, "mean_test_score"]].rename(
            columns={col: p, "mean_test_score": "score"}
        )
        score_tbl = wandb.Table(dataframe=score_plot_df)
        wandb.log({
            f"rnd_{p}_vs_score": wandb.plot.scatter(score_tbl, p, "score", title=f"[RND] {p} vs score")
        })

        std_plot_df = rnd_results_df[[col, "std_test_score"]].rename(
            columns={col: p, "std_test_score": "std"}
        )
        std_tbl = wandb.Table(dataframe=std_plot_df)
        wandb.log({
            f"rnd_{p}_vs_std": wandb.plot.scatter(std_tbl, p, "std", title=f"[RND] {p} vs std")
        })

best_rnd = rnd_clf_search.best_params_
best_rnd_score = rnd_clf_search.best_score_
best_rnd_model = rnd_clf_search.best_estimator_
wandb.log({
    "best_score": best_rnd_score,
    "best_params": best_rnd
})

rnd_model_path = os.path.join(MODEL_DIR, "best_rnd_model.joblib")
joblib.dump(best_rnd_model, rnd_model_path)

artifact = wandb.Artifact("best_rnd_model", type="model")
artifact.add_file(rnd_model_path)
wandb.log_artifact(artifact)

rnd_run.finish()

# Grid search

## TF-idf

### Word

In [ ]:
def make_range(value, pct=0.05, cast_int=False):
    factors = [1 - pct, 1, 1 + pct]

    candidates = [value * f for f in factors]

    if cast_int:
        candidates = [int(round(c)) for c in candidates]

    uniq = []
    for c in candidates:
        if c not in uniq:
            uniq.append(c)

    return uniq

In [ ]:
grid_tfidf_params_word = {
    "tfidf_word__use_idf": [best_rnd["features__tfidf_word__use_idf"]],
    "tfidf_word__sublinear_tf": [best_rnd["features__tfidf_word__sublinear_tf"]],
    "tfidf_word__norm": [best_rnd["features__tfidf_word__norm"]],
    "tfidf_word__ngram_range": [best_rnd["features__tfidf_word__ngram_range"]],
    "tfidf_word__max_df": make_range(best_rnd["features__tfidf_word__max_df"]),
    "tfidf_word__min_df": make_range(best_rnd["features__tfidf_word__min_df"]),
    "tfidf_word__max_features": make_range(
        best_rnd["tfidf_word__max_features"], cast_int=True
    ),
}
grid_tfidf_params_word

In [ ]:
grid_tfidf_search_word = GridSearchCV(
    Pipeline([
        ("tfidf_char", TfidfVectorizer(analyzer="word")), # type: ignore
        ("clf", LinearSVC(
            C=best_rnd["clf__C"],
            random_state=int(os.getenv("RANDOM_SEED", 880055535)),
        )),
    ]),
    grid_tfidf_params_word,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
)

In [ ]:
grid_tfidf_search_word.fit(texts, y_train)

In [ ]:
best_tfidf_word = grid_tfidf_search_word.best_params_
best_tfidf_word

### Char

In [ ]:
grid_tfidf_params_char = {
    "tfidf_char__use_idf": [best_rnd["features__tfidf_char__use_idf"]],
    "tfidf_char__sublinear_tf": [best_rnd["features__tfidf_char__sublinear_tf"]],
    "tfidf_char__norm": [best_rnd["features__tfidf_char__norm"]],
    "tfidf_char__ngram_range": [best_rnd["features__tfidf_char__ngram_range"]],
    "tfidf_char__max_df": make_range(best_rnd["features__tfidf_char__max_df"]),
    "tfidf_char__min_df": make_range(best_rnd["features__tfidf_char__min_df"]),
    "tfidf_char__max_features": make_range(
        best_rnd["tfidf_char__max_features"], cast_int=True
    ),
}
grid_tfidf_params_char

In [ ]:
grid_tfidf_search_char = GridSearchCV(
    Pipeline([
        ("tfidf_char", TfidfVectorizer(analyzer="char")), # type: ignore
        ("clf", LinearSVC(
            C=best_rnd["clf__C"],
            random_state=int(os.getenv("RANDOM_SEED", 880055535)),
        )),
    ]),
    grid_tfidf_params_char,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
)

In [ ]:
grid_tfidf_search_char.fit(texts, y_train)

In [ ]:
best_tfidf_char = grid_tfidf_search_char.best_params_
best_tfidf_char

## SVD

In [ ]:
grid_svd_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("tfidf_char", TfidfVectorizer(analyzer="char")),
        ("tfidf_word", TfidfVectorizer(analyzer="word")), # type: ignore
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))), # type: ignore
])
grid_merged_params_for_svd = {
    **{f"features__{k}": v for k, v in best_tfidf_word.items()},
    **{f"features__{k}": v for k, v in best_tfidf_char.items()},
}
print(grid_merged_params_for_svd)
grid_svd_pipeline.set_params(**grid_merged_params_for_svd)

In [ ]:
grid_svd_params = {
    "svd__n_components": make_range(best_rnd["svd__n_components"], pct=0.1, cast_int=True),
}
grid_svd_params

In [ ]:
grid_svd_search = GridSearchCV(
    grid_svd_pipeline,
    grid_svd_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
)

grid_svd_search.fit(texts, y_train)

In [ ]:
best_svd = grid_svd_search.best_params_
best_svd_score = grid_svd_search.best_score_

## CLF

In [ ]:
grid_clf_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("tfidf_word", TfidfVectorizer(analyzer="word")),
        ("tfidf_char", TfidfVectorizer(analyzer="char")), # type: ignore
    ])),
    ("svd", TruncatedSVD(random_state=int(os.getenv("RANDOM_SEED", 880055535)))),
    ("clf", LinearSVC(random_state=int(os.getenv("RANDOM_SEED", 880055535)))),
])
grid_merged_params_for_clf = {
    **{f"features__{k}": v for k, v in best_tfidf_word.items()},
    **{f"features__{k}": v for k, v in best_tfidf_char.items()},
    **{f"svd__{k}": v for k, v in best_svd.items()},
}
print(grid_merged_params_for_clf)
grid_clf_pipeline.set_params(**grid_merged_params_for_clf)

In [ ]:
grid_clf_params = {
    "clf__C": make_range(best_rnd["clf__C"], pct=0.1),
}
grid_clf_params

In [ ]:
grid_clf_search = GridSearchCV(
    grid_clf_pipeline,
    grid_clf_params,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=5,
)

In [ ]:
grid_clf_search.fit(texts, y_train)

In [ ]:
grid_search = grid_clf_search
grid_params = grid_clf_params

## Logging results

In [ ]:
grid_run = wandb.init(
    project="who-wrote-it-nlp",
    name="hyperparam_search",
    entity="who-wrote-it-nlp",
    job_type="hyperparam_search_grid",
    group="grid_search",
)

stage_results = [
    ("tfidf_char", grid_tfidf_search_char, grid_tfidf_params_char),
    ("tfidf_word", grid_tfidf_search_word, grid_tfidf_params_word),
    ("svd", grid_svd_search, grid_svd_params),
    ("clf", grid_clf_search, grid_clf_params),
]

for stage_name, stage_search, stage_params in stage_results:
    stage_df = pd.DataFrame(stage_search.cv_results_)

    for _, row in stage_df.iterrows():
        params = row["params"]
        wandb.log(
            {
                **params,
                f"{stage_name}_score": row["mean_test_score"],
                f"{stage_name}_std": row["std_test_score"],
                "stage": stage_name,
            },
        )

    wandb.log({f"grid_{stage_name}_cv_results": wandb.Table(dataframe=stage_df)})

    for p in stage_params.keys():
        col = f"param_{p}"
        if col in stage_df.columns and pd.api.types.is_numeric_dtype(stage_df[col]):
            plot_df = stage_df[[col, "mean_test_score"]].rename(
                columns={col: p, "mean_test_score": "score"}
            )
            tbl = wandb.Table(dataframe=plot_df)
            wandb.log({
                f"grid_{stage_name}_{p}_vs_score": wandb.plot.scatter(
                    tbl, p, "score", title=f"[{stage_name.upper()}] {p} vs score"
                )
            })

best_tfidf_char = grid_tfidf_search_char.best_params_
best_tfidf_word = grid_tfidf_search_word.best_params_
best_svd = grid_svd_search.best_params_
best_grid = grid_clf_search.best_params_
best_grid_score = grid_clf_search.best_score_
best_grid_model = grid_clf_search.best_estimator_

wandb.log({
    "best_tfidf_char": best_tfidf_char,
    "best_tfidf_word": best_tfidf_word,
    "best_tfidf_char_score": grid_tfidf_search_char.best_score_,
    "best_tfidf_word_score": grid_tfidf_search_word.best_score_,
    "best_svd": best_svd,
    "best_svd_score": grid_svd_search.best_score_,
    "best_score": best_grid_score,
    "best_params": best_grid,
})

grid_model_path = os.path.join(MODEL_DIR, "best_grid_model.joblib")
joblib.dump(best_grid_model, grid_model_path)

artifact = wandb.Artifact("best_grid_model", type="model")
artifact.add_file(grid_model_path)
wandb.log_artifact(artifact)

grid_run.finish()